# Multi-Head Stance Model — LLM Disagreement Analysis

Trains a shared **DeBERTa-v3-base** encoder with **one Linear classification head per LLM**.
All heads are trained simultaneously on the same batch; the encoder learns a shared representation
that satisfies all four labeling functions at once.

After training:
- **Gradient attribution per head** → which tokens drive each model's decision?
- **Head weight cosine similarity** → how similar are the learned linear mappings?
- **Swap analysis** → what label does head_llama give when applied to deepseek's same chunk?

**Upload** the 4 `chunk_predictions_*.csv` files to `/content/` before running.

**Runtime:** ~30–40 min on T4.

In [ ]:
!pip install -q transformers accelerate scikit-learn scipy matplotlib seaborn

In [ ]:
import os, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from itertools import combinations
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Data Loading

Upload `chunk_predictions_llama33.csv`, `chunk_predictions_deepseekv3.csv`,
`chunk_predictions_qwen25_72b.csv`, `chunk_predictions_mistrallarge_or.csv` to `/content/`.
The notebook works with however many files are present (minimum 2).

In [ ]:
MODELS = {
    'llama33':         'Llama 3.3',
    'deepseekv3':      'DeepSeek V3',
    'qwen25_72b':      'Qwen 2.5 72B',
    'mistrallarge_or': 'Mistral Large',
}
STANCE_SCORE = {
    'dovish': -1.0, 'mostly dovish': -0.5, 'neutral': 0.0,
    'mostly hawkish': 0.5, 'hawkish': 1.0,
}
LABEL_ORDER = ['dovish', 'mostly dovish', 'neutral', 'mostly hawkish', 'hawkish']
LABEL2IDX   = {l: i for i, l in enumerate(LABEL_ORDER)}
IDX2LABEL   = {i: l for l, i in LABEL2IDX.items()}
DIRECTIONAL = ['dovish', 'mostly dovish', 'mostly hawkish', 'hawkish']

loaded = []
all_chunks = {}
for key in MODELS:
    path = f'/content/chunk_predictions_{key}.csv'
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    df = df[df['label'].isin(STANCE_SCORE)].copy()
    df['score'] = df['label'].map(STANCE_SCORE)
    all_chunks[key] = df
    loaded.append(key)
    print(f'  {key}: {len(df):,} chunks')

assert len(loaded) >= 2, 'Need at least 2 chunk_predictions_*.csv files in /content/'
print(f'\nLoaded: {[MODELS[k] for k in loaded]}')

In [ ]:
# Build chunks_wide: inner join on chunk_uid
base_cols = ['chunk_uid', 'bank', 'date', 'text']

chunks_wide = (all_chunks[loaded[0]][base_cols + ['label', 'score']]
               .rename(columns={'label': f'label_{loaded[0]}',
                                'score': f'score_{loaded[0]}'}))
for key in loaded[1:]:
    chunks_wide = chunks_wide.merge(
        all_chunks[key][['chunk_uid', 'label', 'score']].rename(
            columns={'label': f'label_{key}', 'score': f'score_{key}'}),
        on='chunk_uid')

for key in loaded:
    chunks_wide[f'stanced_{key}'] = chunks_wide[f'label_{key}'].isin(DIRECTIONAL)

chunks_wide['n_stanced'] = chunks_wide[[f'stanced_{k}' for k in loaded]].sum(axis=1)
chunks_wide['split']     = chunks_wide['n_stanced'].between(1, len(loaded) - 1)

lbl2ord = {l: i for i, l in enumerate(LABEL_ORDER)}
chunks_wide['max_ordinal_gap'] = chunks_wide.apply(
    lambda r: max(
        abs(lbl2ord.get(r[f'label_{a}'], 2) - lbl2ord.get(r[f'label_{b}'], 2))
        for a, b in combinations(loaded, 2)
    ), axis=1)

print(f'chunks_wide: {len(chunks_wide):,} rows')
print(f'Split (disagreement): {chunks_wide["split"].sum()} ({chunks_wide["split"].mean():.1%})')
print()
for key in loaded:
    dist = chunks_wide[f'label_{key}'].value_counts().reindex(LABEL_ORDER, fill_value=0)
    print(f'{MODELS[key]}: {dist.to_dict()}')

## Dataset and DataLoader

In [ ]:
# 70 / 15 / 15 split by chunk_uid (same chunk -> same split for all heads)
uids = chunks_wide['chunk_uid'].tolist()
random.shuffle(uids)
n = len(uids)
train_uids = set(uids[:int(0.70 * n)])
val_uids   = set(uids[int(0.70 * n):int(0.85 * n)])
test_uids  = set(uids[int(0.85 * n):])
chunks_wide['split_set'] = chunks_wide['chunk_uid'].apply(
    lambda u: 'train' if u in train_uids else ('val' if u in val_uids else 'test'))
print(f'train={len(train_uids)}  val={len(val_uids)}  test={len(test_uids)}')

MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiHeadDataset(Dataset):
    def __init__(self, df, model_keys, max_length=512):
        self.texts     = df['text'].tolist()
        self.labels    = {k: df[f'label_{k}'].map(LABEL2IDX).tolist() for k in model_keys}
        self.chunk_uid = df['chunk_uid'].tolist()
        self.is_split  = df['split'].tolist()
        self.bank      = df['bank'].tolist()
        self.max_length = max_length
        self.keys = model_keys

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         {k: torch.tensor(self.labels[k][idx]) for k in self.keys},
            'chunk_uid':      self.chunk_uid[idx],
            'is_split':       self.is_split[idx],
            'bank':           self.bank[idx],
        }

def make_loader(split_name, batch_size=8, shuffle=True):
    df = chunks_wide[chunks_wide['split_set'] == split_name].reset_index(drop=True)
    return DataLoader(MultiHeadDataset(df, loaded), batch_size=batch_size,
                      shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader('train', batch_size=8,  shuffle=True)
val_loader   = make_loader('val',   batch_size=16, shuffle=False)
test_loader  = make_loader('test',  batch_size=16, shuffle=False)
print(f'Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}')

## Model Definition

```
chunk text -> DeBERTa-v3-base -> [CLS] (768-dim)
                               -> head_llama33       Linear(768->5)
                               -> head_deepseekv3    Linear(768->5)
                               -> head_qwen25_72b    Linear(768->5)
                               -> head_mistrallarge  Linear(768->5)
```

All heads share the encoder. A single forward pass returns logits from all heads simultaneously.

In [ ]:
class MultiHeadStanceModel(nn.Module):
    def __init__(self, encoder_name, model_keys, n_classes=5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size  # 768
        self.heads = nn.ModuleDict({
            key: nn.Linear(hidden, n_classes) for key in model_keys
        })
        self.keys = model_keys

    def forward(self, input_ids, attention_mask):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = enc.last_hidden_state[:, 0, :].float()   # cast to float32; encoder outputs float16 on T4
        logits = {key: self.heads[key](cls) for key in self.keys}
        return logits, cls

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True

model = MultiHeadStanceModel(MODEL_NAME, loaded).to(device)

n_enc   = sum(p.numel() for p in model.encoder.parameters())
n_heads = sum(p.numel() for p in model.heads.parameters())
print(f'Encoder params : {n_enc:,}')
print(f'Head params    : {n_heads:,}  ({len(loaded)} heads x {n_heads//len(loaded):,} each)')
print(f'Total          : {n_enc + n_heads:,}')

## Training

- **Epoch 1**: encoder frozen, heads trained only (lr=1e-3). Warms up the classification heads before touching the encoder weights.
- **Epochs 2–3**: encoder unfrozen (lr=2e-5), heads continue at lr=1e-3.
- **Loss per batch**: sum of CrossEntropy over all heads (one loss per head per example).
- **Gradient accumulation**: 4 steps → effective batch size 32.

In [ ]:
# ── Resume from saved checkpoint (skip retraining) ───────────────────────────
# Option A: load from Google Drive (recommended if you saved there before)
# Option B: upload the .pt file directly from your computer
#
# Set SKIP_TRAINING = True to use a checkpoint; False to train from scratch.

SKIP_TRAINING  = True
CKPT_FROM_DRIVE = True   # False = upload from local disk instead

if SKIP_TRAINING:
    if CKPT_FROM_DRIVE:
        from google.colab import drive
        drive.mount('/drive', force_remount=False)
        CKPT_PATH = '/drive/MyDrive/central_bank_spillovers/multihead/model_multihead_checkpoint.pt'
    else:
        from google.colab import files as colab_files
        print("Upload model_multihead_checkpoint.pt")
        up = colab_files.upload()   # opens a file picker
        CKPT_PATH = list(up.keys())[0]

    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.float().to(device)
    history       = ckpt.get('history', [])
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f"Loaded checkpoint from {CKPT_PATH}")
    print(f"Keys in checkpoint: {list(ckpt.keys())}")
else:
    print("SKIP_TRAINING=False — run the training cell next.")


In [ ]:
if not SKIP_TRAINING:
    E    P    O    C    H    S                   =         3
    G    R    A    D    _    A    C    C         =         4
    W    A    R    M    U    P                   =         1    0    0

    #         D    e    B    E    R    T    a         e    n    c    o    d    e    r         c    a    n         r    u    n         i    n         f    l    o    a    t    1    6         o    n         T    4    ;         h    e    a    d    s         a    r    e         f    l    o    a    t    3    2    .         F    o    r    c    e         e    v    e    r    y    t    h    i    n    g         t    o         f    l    o    a    t    3    2    .
    m    o    d    e    l         =         m    o    d    e    l    .    f    l    o    a    t    (    )

    c    r    i    t    e    r    i    o    n         =         n    n    .    C    r    o    s    s    E    n    t    r    o    p    y    L    o    s    s    (    )

    d    e    f         m    a    k    e    _    o    p    t    i    m    i    z    e    r    (    m    o    d    e    l    ,         e    n    c    o    d    e    r    _    l    r    =    2    e    -    5    ,         h    e    a    d    _    l    r    =    1    e    -    3    )    :
                        r    e    t    u    r    n         t    o    r    c    h    .    o    p    t    i    m    .    A    d    a    m    W    (    [
                                            {    '    p    a    r    a    m    s    '    :         m    o    d    e    l    .    e    n    c    o    d    e    r    .    p    a    r    a    m    e    t    e    r    s    (    )    ,         '    l    r    '    :         e    n    c    o    d    e    r    _    l    r    ,         '    w    e    i    g    h    t    _    d    e    c    a    y    '    :         0    .    0    1    }    ,
                                            {    '    p    a    r    a    m    s    '    :         m    o    d    e    l    .    h    e    a    d    s    .    p    a    r    a    m    e    t    e    r    s    (    )    ,                   '    l    r    '    :         h    e    a    d    _    l    r    ,                        '    w    e    i    g    h    t    _    d    e    c    a    y    '    :         0    .    0    1    }    ,
                        ]    )

    @    t    o    r    c    h    .    n    o    _    g    r    a    d    (    )
    d    e    f         e    v    a    l    u    a    t    e    (    l    o    a    d    e    r    )    :
                        m    o    d    e    l    .    e    v    a    l    (    )
                        a    l    l    _    p    r    e    d    s         =         {    k    :         [    ]         f    o    r         k         i    n         l    o    a    d    e    d    }
                        a    l    l    _    t    r    u    e              =         {    k    :         [    ]         f    o    r         k         i    n         l    o    a    d    e    d    }
                        t    o    t    a    l    _    l    o    s    s         =         0    .    0
                        f    o    r         b    a    t    c    h         i    n         l    o    a    d    e    r    :
                                            i    d    s                   =         b    a    t    c    h    [    '    i    n    p    u    t    _    i    d    s    '    ]    .    t    o    (    d    e    v    i    c    e    )
                                            m    a    s    k              =         b    a    t    c    h    [    '    a    t    t    e    n    t    i    o    n    _    m    a    s    k    '    ]    .    t    o    (    d    e    v    i    c    e    )
                                            l    o    g    i    t    s    _    d    i    c    t    ,         _         =         m    o    d    e    l    (    i    d    s    ,         m    a    s    k    )
                                            l    o    s    s         =         s    u    m    (    c    r    i    t    e    r    i    o    n    (    l    o    g    i    t    s    _    d    i    c    t    [    k    ]    ,         b    a    t    c    h    [    '    l    a    b    e    l    s    '    ]    [    k    ]    .    t    o    (    d    e    v    i    c    e    )    )         f    o    r         k         i    n         l    o    a    d    e    d    )
                                            t    o    t    a    l    _    l    o    s    s         +    =         l    o    s    s    .    i    t    e    m    (    )
                                            f    o    r         k         i    n         l    o    a    d    e    d    :
                                                                a    l    l    _    p    r    e    d    s    [    k    ]    .    e    x    t    e    n    d    (    l    o    g    i    t    s    _    d    i    c    t    [    k    ]    .    a    r    g    m    a    x    (    1    )    .    c    p    u    (    )    .    t    o    l    i    s    t    (    )    )
                                                                a    l    l    _    t    r    u    e    [    k    ]    .    e    x    t    e    n    d    (    b    a    t    c    h    [    '    l    a    b    e    l    s    '    ]    [    k    ]    .    t    o    l    i    s    t    (    )    )
                        a    c    c    s         =         {    k    :         a    c    c    u    r    a    c    y    _    s    c    o    r    e    (    a    l    l    _    t    r    u    e    [    k    ]    ,         a    l    l    _    p    r    e    d    s    [    k    ]    )         f    o    r         k         i    n         l    o    a    d    e    d    }
                        f    1    s              =         {    k    :         f    1    _    s    c    o    r    e    (    a    l    l    _    t    r    u    e    [    k    ]    ,         a    l    l    _    p    r    e    d    s    [    k    ]    ,         a    v    e    r    a    g    e    =    '    m    a    c    r    o    '    ,         z    e    r    o    _    d    i    v    i    s    i    o    n    =    0    )         f    o    r         k         i    n         l    o    a    d    e    d    }
                        r    e    t    u    r    n         t    o    t    a    l    _    l    o    s    s         /         l    e    n    (    l    o    a    d    e    r    )    ,         a    c    c    s    ,         f    1    s

    h    i    s    t    o    r    y         =         {    '    t    r    a    i    n    _    l    o    s    s    '    :         [    ]    ,         '    v    a    l    _    l    o    s    s    '    :         [    ]    ,         '    v    a    l    _    a    c    c    '    :         {    k    :         [    ]         f    o    r         k         i    n         l    o    a    d    e    d    }    }
    b    e    s    t    _    v    a    l    _    l    o    s    s         =         f    l    o    a    t    (    '    i    n    f    '    )
    b    e    s    t    _    s    t    a    t    e                        =         N    o    n    e
    s    c    h    e    d    u    l    e    r                             =         N    o    n    e

    f    o    r         e    p    o    c    h         i    n         r    a    n    g    e    (    E    P    O    C    H    S    )    :
                        i    f         e    p    o    c    h         =    =         0    :
                                            m    o    d    e    l    .    f    r    e    e    z    e    _    e    n    c    o    d    e    r    (    )
                                            o    p    t    i    m    i    z    e    r         =         m    a    k    e    _    o    p    t    i    m    i    z    e    r    (    m    o    d    e    l    ,         e    n    c    o    d    e    r    _    l    r    =    0    .    0    ,         h    e    a    d    _    l    r    =    1    e    -    3    )
                                            p    r    i    n    t    (    '    E    p    o    c    h         1    :         e    n    c    o    d    e    r         f    r    o    z    e    n         —         t    r    a    i    n    i    n    g         h    e    a    d    s         o    n    l    y    '    )
                        e    l    i    f         e    p    o    c    h         =    =         1    :
                                            m    o    d    e    l    .    u    n    f    r    e    e    z    e    _    e    n    c    o    d    e    r    (    )
                                            o    p    t    i    m    i    z    e    r         =         m    a    k    e    _    o    p    t    i    m    i    z    e    r    (    m    o    d    e    l    ,         e    n    c    o    d    e    r    _    l    r    =    2    e    -    5    ,         h    e    a    d    _    l    r    =    1    e    -    3    )
                                            t    o    t    a    l    _    s    t    e    p    s         =         l    e    n    (    t    r    a    i    n    _    l    o    a    d    e    r    )         *         (    E    P    O    C    H    S         -         1    )         /    /         G    R    A    D    _    A    C    C
                                            s    c    h    e    d    u    l    e    r                   =         g    e    t    _    l    i    n    e    a    r    _    s    c    h    e    d    u    l    e    _    w    i    t    h    _    w    a    r    m    u    p    (    o    p    t    i    m    i    z    e    r    ,         W    A    R    M    U    P    ,         t    o    t    a    l    _    s    t    e    p    s    )
                                            p    r    i    n    t    (    '    E    p    o    c    h         2    +    :         e    n    c    o    d    e    r         u    n    f    r    o    z    e    n         (    l    r    =    2    e    -    5    )    '    )

                        m    o    d    e    l    .    t    r    a    i    n    (    )
                        e    p    o    c    h    _    l    o    s    s         =         0    .    0
                        o    p    t    i    m    i    z    e    r    .    z    e    r    o    _    g    r    a    d    (    )

                        f    o    r         s    t    e    p    ,         b    a    t    c    h         i    n         e    n    u    m    e    r    a    t    e    (    t    r    a    i    n    _    l    o    a    d    e    r    )    :
                                            i    d    s              =         b    a    t    c    h    [    '    i    n    p    u    t    _    i    d    s    '    ]    .    t    o    (    d    e    v    i    c    e    )
                                            m    a    s    k         =         b    a    t    c    h    [    '    a    t    t    e    n    t    i    o    n    _    m    a    s    k    '    ]    .    t    o    (    d    e    v    i    c    e    )
                                            l    o    g    i    t    s    _    d    i    c    t    ,         _         =         m    o    d    e    l    (    i    d    s    ,         m    a    s    k    )
                                            l    o    s    s         =         s    u    m    (    c    r    i    t    e    r    i    o    n    (    l    o    g    i    t    s    _    d    i    c    t    [    k    ]    ,         b    a    t    c    h    [    '    l    a    b    e    l    s    '    ]    [    k    ]    .    t    o    (    d    e    v    i    c    e    )    )         f    o    r         k         i    n         l    o    a    d    e    d    )
                                            (    l    o    s    s         /         G    R    A    D    _    A    C    C    )    .    b    a    c    k    w    a    r    d    (    )
                                            e    p    o    c    h    _    l    o    s    s         +    =         l    o    s    s    .    i    t    e    m    (    )

                                            i    f         (    s    t    e    p         +         1    )         %         G    R    A    D    _    A    C    C         =    =         0    :
                                                                n    n    .    u    t    i    l    s    .    c    l    i    p    _    g    r    a    d    _    n    o    r    m    _    (    m    o    d    e    l    .    p    a    r    a    m    e    t    e    r    s    (    )    ,         1    .    0    )
                                                                o    p    t    i    m    i    z    e    r    .    s    t    e    p    (    )
                                                                i    f         s    c    h    e    d    u    l    e    r         i    s         n    o    t         N    o    n    e    :
                                                                                    s    c    h    e    d    u    l    e    r    .    s    t    e    p    (    )
                                                                o    p    t    i    m    i    z    e    r    .    z    e    r    o    _    g    r    a    d    (    )

                                            i    f         (    s    t    e    p         +         1    )         %         1    0    0         =    =         0    :
                                                                p    r    i    n    t    (    f    '              s    t    e    p         {    s    t    e    p    +    1    }    /    {    l    e    n    (    t    r    a    i    n    _    l    o    a    d    e    r    )    }              l    o    s    s    =    {    e    p    o    c    h    _    l    o    s    s    /    (    s    t    e    p    +    1    )    :    .    4    f    }    '    )

                        t    r    a    i    n    _    l    o    s    s         =         e    p    o    c    h    _    l    o    s    s         /         l    e    n    (    t    r    a    i    n    _    l    o    a    d    e    r    )
                        v    a    l    _    l    o    s    s    ,         v    a    l    _    a    c    c    s    ,         v    a    l    _    f    1    s         =         e    v    a    l    u    a    t    e    (    v    a    l    _    l    o    a    d    e    r    )
                        h    i    s    t    o    r    y    [    '    t    r    a    i    n    _    l    o    s    s    '    ]    .    a    p    p    e    n    d    (    t    r    a    i    n    _    l    o    s    s    )
                        h    i    s    t    o    r    y    [    '    v    a    l    _    l    o    s    s    '    ]    .    a    p    p    e    n    d    (    v    a    l    _    l    o    s    s    )
                        f    o    r         k         i    n         l    o    a    d    e    d    :
                                            h    i    s    t    o    r    y    [    '    v    a    l    _    a    c    c    '    ]    [    k    ]    .    a    p    p    e    n    d    (    v    a    l    _    a    c    c    s    [    k    ]    )

                        p    r    i    n    t    (    f    '    \    n    E    p    o    c    h         {    e    p    o    c    h    +    1    }    :         t    r    a    i    n    =    {    t    r    a    i    n    _    l    o    s    s    :    .    4    f    }              v    a    l    =    {    v    a    l    _    l    o    s    s    :    .    4    f    }    '    )
                        f    o    r         k         i    n         l    o    a    d    e    d    :
                                            p    r    i    n    t    (    f    '              {    M    O    D    E    L    S    [    k    ]    :    <    2    2    }    :         a    c    c    =    {    v    a    l    _    a    c    c    s    [    k    ]    :    .    3    f    }              F    1    =    {    v    a    l    _    f    1    s    [    k    ]    :    .    3    f    }    '    )

                        i    f         v    a    l    _    l    o    s    s         <         b    e    s    t    _    v    a    l    _    l    o    s    s    :
                                            b    e    s    t    _    v    a    l    _    l    o    s    s         =         v    a    l    _    l    o    s    s
                                            b    e    s    t    _    s    t    a    t    e         =         {    k    :         v    .    c    p    u    (    )    .    c    l    o    n    e    (    )         f    o    r         k    ,         v         i    n         m    o    d    e    l    .    s    t    a    t    e    _    d    i    c    t    (    )    .    i    t    e    m    s    (    )    }
                                            p    r    i    n    t    (    '              -    >         b    e    s    t         c    h    e    c    k    p    o    i    n    t    '    )

    m    o    d    e    l    .    l    o    a    d    _    s    t    a    t    e    _    d    i    c    t    (    b    e    s    t    _    s    t    a    t    e    )
    p    r    i    n    t    (    f    '    \    n    T    r    a    i    n    i    n    g         c    o    m    p    l    e    t    e    .         B    e    s    t         v    a    l         l    o    s    s    :         {    b    e    s    t    _    v    a    l    _    l    o    s    s    :    .    4    f    }    '    )

    #         P    l    o    t         t    r    a    i    n    i    n    g         c    u    r    v    e    s
    f    i    g    ,         a    x    e    s         =         p    l    t    .    s    u    b    p    l    o    t    s    (    1    ,         2    ,         f    i    g    s    i    z    e    =    (    1    2    ,         4    )    )
    a    x    e    s    [    0    ]    .    p    l    o    t    (    h    i    s    t    o    r    y    [    '    t    r    a    i    n    _    l    o    s    s    '    ]    ,         l    a    b    e    l    =    '    t    r    a    i    n    '    )
    a    x    e    s    [    0    ]    .    p    l    o    t    (    h    i    s    t    o    r    y    [    '    v    a    l    _    l    o    s    s    '    ]    ,                   l    a    b    e    l    =    '    v    a    l    '    )
    a    x    e    s    [    0    ]    .    s    e    t    _    t    i    t    l    e    (    '    L    o    s    s    '    )
    a    x    e    s    [    0    ]    .    l    e    g    e    n    d    (    )
    f    o    r         k         i    n         l    o    a    d    e    d    :
                        a    x    e    s    [    1    ]    .    p    l    o    t    (    h    i    s    t    o    r    y    [    '    v    a    l    _    a    c    c    '    ]    [    k    ]    ,         l    a    b    e    l    =    M    O    D    E    L    S    [    k    ]    )
    a    x    e    s    [    1    ]    .    s    e    t    _    t    i    t    l    e    (    '    V    a    l         A    c    c    u    r    a    c    y         p    e    r         H    e    a    d    '    )
    a    x    e    s    [    1    ]    .    l    e    g    e    n    d    (    f    o    n    t    s    i    z    e    =    8    )
    p    l    t    .    t    i    g    h    t    _    l    a    y    o    u    t    (    )
    p    l    t    .    s    a    v    e    f    i    g    (    '    t    r    a    i    n    i    n    g    _    c    u    r    v    e    s    .    p    n    g    '    ,         d    p    i    =    1    2    0    )
    p    l    t    .    s    h    o    w    (    )

In [ ]:
# ── Save checkpoint to Drive immediately after training ──────────────────────
# Run this right after the training cell. If the Colab runtime dies later,
# you can reload the checkpoint and skip retraining.

from google.colab import drive
drive.mount('/drive', force_remount=False)

DRIVE_DIR = '/drive/MyDrive/central_bank_spillovers/multihead'
os.makedirs(DRIVE_DIR, exist_ok=True)

CKPT_PATH = os.path.join(DRIVE_DIR, 'model_multihead_checkpoint.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'loaded_keys':      loaded,
    'label2idx':        LABEL2IDX,
    'model_name':       MODEL_NAME,
    'history':          history,
    'best_val_loss':    best_val_loss,
}, CKPT_PATH)
print(f'Checkpoint saved to Drive: {CKPT_PATH}')

# ── To reload later without retraining ───────────────────────────────────────
# ckpt = torch.load(CKPT_PATH, map_location=device)
# loaded = ckpt['loaded_keys']
# model  = MultiHeadStanceModel(ckpt['model_name'], loaded).to(device)
# model.load_state_dict(ckpt['model_state_dict'])
# model.float()
# model.eval()
# print('Checkpoint loaded from Drive.')

## Evaluation — Per-Head Metrics

In [ ]:
test_loss, test_accs, test_f1s = evaluate(test_loader)
print('=== Test Set ===')
rows = []
for k in loaded:
    print(f'  {MODELS[k]:<22}: acc={test_accs[k]:.3f}  macro-F1={test_f1s[k]:.3f}')
    rows.append({'model_key': k, 'model_name': MODELS[k],
                 'accuracy': test_accs[k], 'macro_f1': test_f1s[k]})
pd.DataFrame(rows).to_csv('per_head_metrics.csv', index=False)

# Confusion matrices
model.eval()
all_preds = {k: [] for k in loaded}
all_true  = {k: [] for k in loaded}
with torch.no_grad():
    for batch in test_loader:
        ids, mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
        logits_dict, _ = model(ids, mask)
        for k in loaded:
            all_preds[k].extend(logits_dict[k].argmax(1).cpu().tolist())
            all_true[k].extend(batch['labels'][k].tolist())

short_labels = ['dov', 'm-dov', 'neu', 'm-hawk', 'hawk']
fig, axes = plt.subplots(1, len(loaded), figsize=(5 * len(loaded), 5))
if len(loaded) == 1:
    axes = [axes]
for ax, k in zip(axes, loaded):
    cm = confusion_matrix(all_true[k], all_preds[k], labels=list(range(5)))
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=short_labels, yticklabels=short_labels)
    ax.set_title(MODELS[k])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.suptitle('Confusion Matrices per Head (Test Set)', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## Gradient Attribution per Head

For each split chunk in the test set, compute
$\|\partial \hat{y}_k / \partial e_t\|_2$ for each token $t$ and each head $k$,
where $e_t$ is the input embedding of token $t$ and $\hat{y}_k$ is the predicted-class logit.

Averaging across chunks gives a per-head token importance profile.
The **delta chart** shows which tokens matter *more* to one model than another.

In [ ]:
STOP_WORDS = {
    'the','and','of','to','in','is','that','for','it','we','on','are','not',
    'this','with','from','have','they','been','will','when','their','also',
    'more','which','were','into','than','then','what','about','some','there',
    'could','would','does','overall','text','excerpt','these','those','most',
    'while','very','such','well','both','because','suggests','indicates',
    'central','bank','monetary','policy','signals','stance','signal',
    'however','although','despite','clear','rather','given','think',
}

def get_token_importance(text, head_key):
    """Return (token_list, importance_array) for one text and one head."""
    model.eval()
    enc = tokenizer(text, max_length=512, truncation=True,
                    padding='max_length', return_tensors='pt')
    input_ids      = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)

    captured = {}
    def hook(module, inp, out):
        captured['emb'] = out
        out.retain_grad()

    # DebertaV2Model (v3) embeds at .embeddings.word_embeddings directly
    handle = model.encoder.embeddings.word_embeddings.register_forward_hook(hook)

    logits_dict, _ = model(input_ids, attention_mask)
    pred_cls = logits_dict[head_key].argmax(1).item()
    logits_dict[head_key][0, pred_cls].backward()
    handle.remove()

    emb = captured.get('emb')
    if emb is None or emb.grad is None:
        return None, None

    importance = emb.grad[0].norm(dim=-1).detach().cpu().numpy()
    tokens     = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())
    mask_arr   = attention_mask[0].cpu().numpy().astype(bool)
    return [t for t, m in zip(tokens, mask_arr) if m], importance[mask_arr]

# Aggregate over test-set split chunks
test_split = chunks_wide[
    (chunks_wide['split_set'] == 'test') & chunks_wide['split']
].head(200).reset_index(drop=True)

print(f'Running attribution on {len(test_split)} split chunks...')
token_importance = {k: defaultdict(list) for k in loaded}

for i, row in test_split.iterrows():
    text = str(row['text'])
    model.zero_grad()
    for key in loaded:
        tokens, importance = get_token_importance(text, key)
        if tokens is None:
            continue
        for tok, imp in zip(tokens, importance):
            clean = tok.replace('▁', '').lower().strip()
            if len(clean) > 2 and clean not in STOP_WORDS and clean.isalpha():
                token_importance[key][clean].append(float(imp))
    if (i + 1) % 25 == 0:
        print(f'  {i+1}/{len(test_split)}')

# Average: keep tokens seen in >=5 chunks
avg_imp = {
    key: {tok: np.mean(vals) for tok, vals in token_importance[key].items() if len(vals) >= 5}
    for key in loaded
}
print('Done.')
for key in loaded:
    top5 = sorted(avg_imp[key].items(), key=lambda x: -x[1])[:5]
    print(f'  {MODELS[key]}: {top5}')

In [ ]:
TOP_N  = 25
COLORS = ['#C44E52', '#4C72B0', '#55A868', '#DD8452']

# Union of top-N tokens across all heads
all_top = set()
for key in loaded:
    top_toks = sorted(avg_imp[key].items(), key=lambda x: -x[1])[:TOP_N]
    all_top.update(t for t, _ in top_toks)
all_top = sorted(
    all_top,
    key=lambda t: max(avg_imp[k].get(t, 0) for k in loaded),
    reverse=True
)[:TOP_N]

n_cols = len(loaded) + 1   # one per head + delta
fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 9))

for ax, key, col in zip(axes[:len(loaded)], loaded, COLORS):
    vals = [avg_imp[key].get(t, 0) for t in all_top]
    ax.barh(all_top, vals, color=col, alpha=0.85)
    ax.set_title(f'{MODELS[key]}\ntoken importance (gradient)', fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Mean ||grad||')

# Delta: head[0] minus head[1]
ax_d = axes[-1]
if len(loaded) >= 2:
    delta = [
        avg_imp[loaded[0]].get(t, 0) - avg_imp[loaded[1]].get(t, 0)
        for t in all_top
    ]
    bar_cols = [COLORS[0] if d > 0 else COLORS[1] for d in delta]
    ax_d.barh(all_top, delta, color=bar_cols, alpha=0.85)
    ax_d.axvline(0, color='black', linewidth=0.8)
    n0 = MODELS[loaded[0]].split()[0]
    n1 = MODELS[loaded[1]].split()[0]
    ax_d.set_title(f'Delta: {n0} \u2212 {n1}\n(+{n0} more important | +{n1} more important)', fontsize=10)
    ax_d.invert_yaxis()
    ax_d.set_xlabel('Importance difference')
else:
    ax_d.axis('off')

plt.suptitle('Per-Head Gradient Attribution on Split Chunks (Test Set)', fontsize=12)
plt.tight_layout()
plt.savefig('head_gradient_attribution.png', dpi=130, bbox_inches='tight')
plt.show()

## Head Weight Analysis

Each head is `Linear(768, 5)` — its weight matrix is directly interpretable.
Cosine similarity between flattened weight matrices tells us how similar two models'
learned linear mappings are. If Llama and Mistral have high cosine similarity,
their labeling functions are nearly identical linear projections of the same CLS embedding.

In [ ]:
head_weights = {}
for key in loaded:
    w = model.heads[key].weight.detach().cpu().numpy()  # (5, 768)
    head_weights[key] = w.flatten()                     # (3840,)

names = [MODELS[k] for k in loaded]
W     = np.stack([head_weights[k] for k in loaded])    # (n_heads, 3840)
cos   = cosine_similarity(W)                            # (n_heads, n_heads)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cos, annot=True, fmt='.3f', cmap='RdYlGn',
            xticklabels=names, yticklabels=names,
            vmin=-1, vmax=1, center=0, ax=axes[0])
axes[0].set_title('Head Weight Cosine Similarity\n(higher = more similar labeling function)')

if len(loaded) >= 3:
    pca    = PCA(n_components=2)
    coords = pca.fit_transform(W)
    for i, (name, key) in enumerate(zip(names, loaded)):
        axes[1].scatter(coords[i, 0], coords[i, 1], s=250,
                        color=COLORS[i % len(COLORS)], zorder=3)
        axes[1].annotate(name, (coords[i, 0], coords[i, 1]),
                         textcoords='offset points', xytext=(8, 4), fontsize=10)
    axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%} var)')
    axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%} var)')
    axes[1].set_title('PCA of Head Weights\n(distance ≈ difference in learned labeling function)')
else:
    axes[1].axis('off')
    axes[1].text(0.5, 0.5, 'PCA requires >=3 heads',
                 ha='center', va='center', transform=axes[1].transAxes)

plt.suptitle('Head Weight Analysis', fontsize=12)
plt.tight_layout()
plt.savefig('head_weight_similarity.png', dpi=130, bbox_inches='tight')
plt.show()

print('Pairwise cosine similarities:')
for i, ki in enumerate(loaded):
    for j, kj in enumerate(loaded):
        if j > i:
            print(f'  {MODELS[ki]} vs {MODELS[kj]}: {cos[i,j]:.3f}')

## Swap Analysis on Split Chunks

For each split chunk in the test set, the encoder runs **once** and all 4 heads see the
**identical CLS representation**. The only thing that differs between predictions is the head.

This directly answers: if we give Llama's head DeepSeek's internal representation of a chunk,
does the label change? If yes, the heads have learned genuinely different linear projections.
If no, the disagreement is already encoded in the CLS representation itself.

In [ ]:
model.eval()
test_split_df = chunks_wide[
    (chunks_wide['split_set'] == 'test') & chunks_wide['split']
].reset_index(drop=True)
test_split_dataset = MultiHeadDataset(test_split_df, loaded, max_length=512)
test_split_loader  = DataLoader(test_split_dataset, batch_size=16, shuffle=False)

swap_rows = []
with torch.no_grad():
    for batch in test_split_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        logits_dict, _ = model(ids, mask)
        for i in range(len(batch['chunk_uid'])):
            row = {
                'chunk_uid': batch['chunk_uid'][i],
                'bank':      batch['bank'][i],
            }
            for key in loaded:
                probs = torch.softmax(logits_dict[key][i], dim=0).cpu().numpy()
                row[f'pred_{key}']      = IDX2LABEL[probs.argmax()]
                row[f'true_{key}']      = IDX2LABEL[batch['labels'][key][i].item()]
                row[f'p_neutral_{key}'] = float(probs[LABEL2IDX['neutral']])
                row[f'p_stanced_{key}'] = float(1 - probs[LABEL2IDX['neutral']])
            row['n_unique_preds'] = len(set(row[f'pred_{k}'] for k in loaded))
            swap_rows.append(row)

swap_df = pd.DataFrame(swap_rows)
print(f'Split chunks in test set: {len(swap_df)}')

print(f'\nHead agreement on split chunks:')
print(swap_df['n_unique_preds'].value_counts().sort_index().to_string())

if len(loaded) >= 2:
    print(f'\nPairwise prediction disagreement rate (heads on same CLS repr):')
    for ka, kb in combinations(loaded, 2):
        rate = (swap_df[f'pred_{ka}'] != swap_df[f'pred_{kb}']).mean()
        print(f'  {MODELS[ka]} vs {MODELS[kb]}: {rate:.1%}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
p_neutral = [swap_df[f'p_neutral_{k}'].mean() for k in loaded]
bars = ax.bar([MODELS[k].split()[0] for k in loaded], p_neutral,
              color=COLORS[:len(loaded)], alpha=0.85)
ax.set_ylim(0, 1)
ax.set_ylabel('Mean P(neutral)')
ax.set_title('Neutral probability per head on split chunks\n(same CLS representation)')
for bar, v in zip(bars, p_neutral):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.2f}', ha='center', fontsize=10)

ax = axes[1]
swap_df['n_unique_preds'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color='#4C72B0', alpha=0.85, rot=0)
ax.set_xlabel('Unique predictions across heads')
ax.set_ylabel('Chunks')
ax.set_title('Head consensus on split chunks\n(1=all agree, higher=more disagreement)')

plt.suptitle('Swap Analysis — Split Chunks (Test Set)', fontsize=12)
plt.tight_layout()
plt.savefig('swap_analysis_multihead.png', dpi=130, bbox_inches='tight')
plt.show()

## Save and Download All Outputs

In [ ]:
import shutil

# ── Copy all outputs to Drive ─────────────────────────────────────────────────
outputs = [
    'per_head_metrics.csv',
    'training_curves.png',
    'confusion_matrices.png',
    'head_gradient_attribution.png',
    'head_weight_similarity.png',
    'swap_analysis_multihead.png',
    'swap_analysis_results.csv',
    'model_multihead_checkpoint.pt',
]

# Drive should already be mounted from the post-training cell; remount if not
if not os.path.exists('/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/drive', force_remount=False)

DRIVE_DIR = '/drive/MyDrive/central_bank_spillovers/multihead'
os.makedirs(DRIVE_DIR, exist_ok=True)

print('Copying to Drive...')
for fname in outputs:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(DRIVE_DIR, fname))
        print(f'  -> Drive: {fname}')
    else:
        print(f'  (skipped, not found): {fname}')
print(f'\nAll outputs at: {DRIVE_DIR}')

# ── Also download locally ─────────────────────────────────────────────────────
from google.colab import files
print('\nDownloading to local machine...')
for fname in outputs:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  Downloaded: {fname}')

## Linear Probe — Predicting Disagreement from CLS Representations

Train a logistic regression on the frozen DeBERTa CLS vectors to predict whether a chunk
will generate model disagreement. If the probe works, disagreement is linearly encoded in
the shared representation — meaning it is predictable from the text alone, before any LLM sees it.

Apply the probe to every chunk in the corpus to get a **disagreement risk score over time**:
a direct map of where model choice most affects downstream spillover estimates.

In [ ]:
# Extract CLS vectors for all chunks from the trained encoder
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

model.eval()
all_df      = chunks_wide.reset_index(drop=True)
all_dataset = MultiHeadDataset(all_df, loaded, max_length=512)
all_loader_cls = DataLoader(all_dataset, batch_size=32, shuffle=False,
                            num_workers=2, pin_memory=True)

cls_list = []
with torch.no_grad():
    for batch in all_loader_cls:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        _, cls = model(ids, mask)
        cls_list.append(cls.cpu().numpy())

X_all = np.vstack(cls_list)            # (n_chunks, 768)
y_all = all_df["split"].astype(int).values   # 1 = disagreement chunk
split_sets = all_df["split_set"].values
print(f"CLS matrix: {X_all.shape}  |  disagreement chunks: {y_all.sum()} / {len(y_all)}")

In [ ]:
# Train logistic regression probe on training split, evaluate on test split
train_mask = split_sets == "train"
test_mask  = split_sets == "test"

scaler     = StandardScaler()
X_train_s  = scaler.fit_transform(X_all[train_mask])
X_test_s   = scaler.transform(X_all[test_mask])
y_train, y_test = y_all[train_mask], y_all[test_mask]

probe = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")
probe.fit(X_train_s, y_train)

y_pred = probe.predict(X_test_s)
y_prob = probe.predict_proba(X_test_s)[:, 1]

print("Linear probe: predicting chunk-level disagreement from DeBERTa CLS vector")
print(classification_report(y_test, y_pred,
      target_names=["consensus", "split"], digits=3))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.3f}")
print()
print("Interpretation: AUC > 0.5 means disagreement is linearly encoded in the")
print("shared representation -- predictable from the text before any LLM scores it.")

In [ ]:
# Score every chunk and plot disagreement risk over time
X_all_s = scaler.transform(X_all)
all_df["disagree_risk"] = probe.predict_proba(X_all_s)[:, 1]

# Date column may be stored as integer YYYYMMDD — parse explicitly
all_df["date"] = pd.to_datetime(
    chunks_wide.reset_index(drop=True)["date"].astype(str), format="%Y%m%d")
meeting_risk = (all_df.groupby(["bank", "date"])["disagree_risk"]
                .mean().reset_index().sort_values(["bank", "date"]))

banks = sorted(meeting_risk["bank"].unique())
fig, axes = plt.subplots(len(banks), 1, figsize=(14, 3.5 * len(banks)), sharex=True)
if len(banks) == 1:
    axes = [axes]

for ax, bank in zip(axes, banks):
    bdf = meeting_risk[meeting_risk["bank"] == bank]
    ax.fill_between(bdf["date"], bdf["disagree_risk"], alpha=0.25, color="#C44E52")
    ax.plot(bdf["date"], bdf["disagree_risk"], color="#C44E52", alpha=0.9, linewidth=1)
    ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, label="decision boundary")
    ax.set_ylabel("P(disagreement)")
    ax.set_ylim(0, 1)
    ax.set_title(f"{bank}")

plt.xlabel("Date")
plt.suptitle(
    "Disagreement Risk Score over Time (Linear Probe on DeBERTa CLS)
"
    "Peaks = where model choice most affects estimated communication shocks",
    fontsize=11)
plt.tight_layout()
plt.savefig("linear_probe_risk_timeline.png", dpi=130, bbox_inches="tight")
plt.show()

all_df[["chunk_uid", "bank", "date", "split", "disagree_risk"]].to_csv(
    "linear_probe_scores.csv", index=False)
meeting_risk.to_csv("linear_probe_meeting_risk.csv", index=False)
print("Saved: linear_probe_risk_timeline.png, linear_probe_scores.csv, linear_probe_meeting_risk.csv")

# Copy to Drive if mounted
if os.path.exists("/drive/MyDrive"):
    import shutil
    DRIVE_DIR = "/drive/MyDrive/central_bank_spillovers/multihead"
    for fname in ["linear_probe_risk_timeline.png",
                  "linear_probe_scores.csv", "linear_probe_meeting_risk.csv"]:
        if os.path.exists(fname):
            shutil.copy(fname, os.path.join(DRIVE_DIR, fname))
    print("Copied to Drive.")
